In [1]:
# %%writefile util/tools.py
# import collections
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# import time
# from copy import deepcopy
# import os


# plt.switch_backend('agg')


# def load_model_compile(model, model_pth, device, strict=False):
#     model = model.to(device)
#     origin_dict = torch.load(model_pth, map_location=device)
#     state_dict = collections.OrderedDict()
#     torch2_model_prefix = '_orig_mod.'
#     offset2 = len(torch2_model_prefix)
#     if list(origin_dict.keys())[0].startswith(torch2_model_prefix) and \
#             not list(model.state_dict().keys())[0].startswith(torch2_model_prefix):
#         for key, value in origin_dict.items():
#             state_dict[key[offset2: len(key)]] = value
#         model.load_state_dict(state_dict, strict=strict)
#     else:
#         model.load_state_dict(origin_dict, strict=strict)
#     return model


# def remove_state_key_prefix(origin_dict, model, prefix='_orig_mod.'):
#     if isinstance(prefix, list):
#         for p in prefix:
#             origin_dict = remove_state_key_prefix(origin_dict, model, prefix=p)
#         return origin_dict

#     if list(origin_dict.keys())[0].startswith(prefix) and \
#             not list(model.state_dict().keys())[0].startswith(prefix):
#         state_dict = collections.OrderedDict()
#         offset2 = len(prefix)
#         for key, value in origin_dict.items():
#             state_dict[key[offset2: len(key)]] = value
#         return state_dict
#     else:
#         return origin_dict


# def instance_norm(ts, dim):
#     mu = ts.mean(dim, keepdims=True)
#     ts = ts - mu
#     var = ((ts ** 2).mean(dim, keepdims=True) + 1e-8) ** 0.5
#     return ts / var


# def adjust_learning_rate(optimizer, scheduler, epoch, args, printout=True):
#     # lr = args.learning_rate * (0.2 ** (epoch // 2))
#     if args.lradj == 'type1':
#         lr_adjust = {epoch: args.learning_rate * (0.5 ** ((epoch - 1) // 1))}
#     elif args.lradj == 'type2':
#         lr_adjust = {
#             2: 5e-5, 4: 1e-5, 6: 5e-6, 8: 1e-6,
#             10: 5e-7, 15: 1e-7, 20: 5e-8
#         }
#     elif args.lradj == 'type3':
#         lr_adjust = {epoch: args.learning_rate if epoch < 3 else args.learning_rate * (0.9 ** ((epoch - 3) // 1))}
#     elif args.lradj == 'type4':
#         lr_adjust = {epoch: args.learning_rate * (0.5 ** ((epoch) // 1))}
#     elif args.lradj == 'type5':
#         lr_adjust = {epoch: args.learning_rate if epoch < 3 else args.learning_rate * (0.95 ** ((epoch - 3) // 1))}
#     elif args.lradj == 'every5':
#         lr_adjust = {epoch: args.learning_rate * (0.5 ** ((epoch - 1) // 5))}
#     elif args.lradj == 'warmup':
#         lr_adjust = {epoch: args.learning_rate * (0.5 ** ((min(epoch, args.warmup_epochs) - 1) // 1))}
#     elif args.lradj == 'constant':
#         lr_adjust = {epoch: args.learning_rate}
#     elif args.lradj == '3':
#         lr_adjust = {epoch: args.learning_rate if epoch < 10 else args.learning_rate*0.1}
#     elif args.lradj == '4':
#         lr_adjust = {epoch: args.learning_rate if epoch < 15 else args.learning_rate*0.1}
#     elif args.lradj == '5':
#         lr_adjust = {epoch: args.learning_rate if epoch < 25 else args.learning_rate*0.1}
#     elif args.lradj == '6':
#         lr_adjust = {epoch: args.learning_rate if epoch < 5 else args.learning_rate*0.1}
#     elif args.lradj == 'TST':
#         lr_adjust = {epoch: scheduler.get_last_lr()[0]}
#     elif args.lradj == 'Crossformer':
#         lr_adjust = {2: args.learning_rate * 0.5 ** 1, 4: args.learning_rate * 0.5 ** 2,
#                      6: args.learning_rate * 0.5 ** 3, 8: args.learning_rate * 0.5 ** 4,
#                      10: args.learning_rate * 0.5 ** 5}
#     else:
#         lr_adjust = {}

#     if epoch in lr_adjust.keys():
#         lr = lr_adjust[epoch]
#         for param_group in optimizer.param_groups:
#             param_group['lr'] = lr
#         if printout: print('Updating learning rate to {}'.format(lr))



# class EarlyStopping:
#     def __init__(self, patience=7, verbose=False, delta=0):
#         self.patience = patience
#         self.verbose = verbose
#         self.counter = 0
#         self.best_score = None
#         self.early_stop = False
#         self.val_loss_min = float('inf')
#         self.delta = delta
#         self.best_checkpoint = None

#     def __call__(self, val_loss, model, path):
#         score = -val_loss
#         if self.best_score is None:
#             self.best_score = score
#             self.best_checkpoint = deepcopy(model.state_dict())
#             # self.save_checkpoint(val_loss, model, path)
#         elif score < self.best_score + self.delta:
#             self.counter += 1
#             print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
#             if self.counter >= self.patience:
#                 self.early_stop = True
#         else:
#             self.best_score = score
#             self.best_checkpoint = deepcopy(model.state_dict())
#             # self.save_checkpoint(val_loss, model, path)
#             self.counter = 0

#     def save_checkpoint(self, val_loss, model, path):
#         if self.verbose:
#             print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
#         if not os.path.exists(path):
#             os.makedirs(path)

#         self.best_checkpoint = deepcopy(model.state_dict())
#         torch.save(self.best_checkpoint, os.path.join(path, 'checkpoint.pth'))
#         self.val_loss_min = val_loss


# class dotdict(dict):
#     """dot.notation access to dictionary attributes"""
#     __getattr__ = dict.get
#     __setattr__ = dict.__setitem__
#     __delattr__ = dict.__delitem__


# class StandardScaler():
#     def __init__(self, mean, std):
#         self.mean = mean
#         self.std = std

#     def transform(self, data):
#         return (data - self.mean) / self.std

#     def inverse_transform(self, data):
#         return (data * self.std) + self.mean


# def visual(true, preds=None, name='./pic/test.pdf'):
#     """
#     Results visualization
#     """
#     plt.figure()
#     plt.plot(true, label='GroundTruth', linewidth=2)
#     if preds is not None:
#         plt.plot(preds, label='Prediction', linewidth=2)
#     plt.legend()
#     plt.savefig(name, bbox_inches='tight')

# def test_params_flop(model, x_shape, x=None):
#     """
#     If you want to test former's flop, you need to give default value to inputs in model.forward(), the following code can only pass one argument to forward()
#     """
#     import logging
#     logging.disable()
#     import warnings
#     warnings.filterwarnings('ignore')
#     from deepspeed.profiling.flops_profiler import get_model_profile, flops_to_string, number_to_string
#     flops, macs, params = get_model_profile(model=model, input_shape=x_shape, detailed=False, print_profile=False,
#                                             module_depth=0, top_modules=0, as_string=False)
#     print('FLOPS:', flops_to_string(flops, precision=3))

#     model_params = sum([param.nelement() for param in model.parameters()])
#     print('Model parameter count: {}'.format(model_params))
#     model_params = sum([param.nelement() if param.requires_grad else 0 for param in model.parameters()])
#     print('Trainable parameter count: {}'.format(model_params))

#     # from ptflops import get_model_complexity_info
#     # with torch.cuda.device(0):
#     #     macs, params = get_model_complexity_info(model.cuda(), x_shape[1:], as_strings=True, print_per_layer_stat=False)
#     #     print('{:<30}  {:<8}'.format('Computational complexity:', macs))
#     #     print('{:<30}  {:<8}'.format('Number of parameters:', params))

#     # import torchinfo
#     # torchinfo.summary(model, x_shape, depth=1)





In [2]:
%%writefile run.py
import argparse
import datetime
import gc

import settings
from data_provider import data_loader

cur_sec = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print(cur_sec)

from pprint import pprint

import random

import exp as exps
from exp import *

from settings import data_settings


def str_to_bool(value):
    if isinstance(value, bool):
        return value
    if value.lower() in {'false', 'f', '0', 'no', 'n'}:
        return False
    elif value.lower() in {'true', 't', '1', 'yes', 'y'}:
        return True
    raise ValueError(f'{value} is not a valid boolean value')


parser = argparse.ArgumentParser()

# basic config
parser.add_argument('--train_only', action='store_true', default=False,
                    help='perform training on full input dataset without validation and testing')
parser.add_argument('--wo_test', action='store_true', default=False, help='only valid, not test')
parser.add_argument('--wo_valid', action='store_true', default=False, help='only test')
# parser.add_argument('--model_id', type=str, required=True, default='test', help='model id')
parser.add_argument('--only_test', action='store_true', default=False)
parser.add_argument('--do_valid', action='store_true', default=False)
parser.add_argument('--model', type=str, required=True, default='PatchTST')
parser.add_argument('--override_hyper', action='store_true', default=True, help='Override hyperparams by setting.py')
parser.add_argument('--compile', action='store_true', default=False, help='Compile the model by Pytorch 2.0')
parser.add_argument('--reduce_bs', type=str_to_bool, default=False,
                    help='Override batch_size in hyperparams by setting.py')
parser.add_argument('--normalization', type=str, default=None)
parser.add_argument('--checkpoints', type=str, default='./checkpoints/', help='location of model checkpoints')
parser.add_argument('--tag', type=str, default='')

# online
parser.add_argument('--online_method', type=str, default=None)
parser.add_argument('--skip', type=str, default=None)
parser.add_argument('--online_learning_rate', type=float, default=None)
parser.add_argument('--val_online_lr', action='store_true', default=True)
parser.add_argument('--diff_online_lr', action='store_true', default=False)
parser.add_argument('--save_opt', action='store_true', default=True)
parser.add_argument('--leakage', action='store_true', default=False)
parser.add_argument('--debug', action='store_true', default=False)
parser.add_argument('--pretrain', action='store_true', default=False)
parser.add_argument('--freeze', action='store_true', default=False)

# Proceed
parser.add_argument('--act', type=str, default='sigmoid', help='activation')
parser.add_argument('--tune_mode', type=str, default='down_up')
parser.add_argument('--ema', type=float, default=0, help='')
parser.add_argument('--concept_dim', type=int, default=200)
parser.add_argument('--bottleneck_dim', type=int, default=32, help='')
parser.add_argument('--individual_generator', action='store_true', default=False)
parser.add_argument('--share_encoder', action='store_true', default=False)
parser.add_argument('--use_mean', type=str_to_bool, default=True)
parser.add_argument('--joint_update_valid', action='store_true', default=False)
parser.add_argument('--comment', type=str, default='')
parser.add_argument('--wo_clip', action='store_true', default=False)

# OneNet
parser.add_argument('--learning_rate_w', type=float, default=0.001, help='optimizer learning rate')
parser.add_argument('--learning_rate_bias', type=float, default=0.001, help='optimizer learning rate')

# data loader
parser.add_argument('--border_type', type=str, default='online', help='set any other value for traditional data splits')
parser.add_argument('--root_path', type=str, default='./dataset/', help='root path of the data file')
parser.add_argument('--dataset', type=str, default='ETTh1', help='data file')
parser.add_argument('--features', type=str, default='M',
                    help='forecasting task, options:[M, S, MS]; M:multivariate predict multivariate, S:univariate predict univariate, MS:multivariate predict univariate')
parser.add_argument('--target', type=str, default='OT', help='target feature in S or MS task')
parser.add_argument('--freq', type=str, default='h',
                    help='freq for time features encoding, options:[s:secondly, t:minutely, h:hourly, d:daily, b:business days, w:weekly, m:monthly], you can also use more detailed freq like 15min or 3h')
parser.add_argument('--wrap_data_class', type=list, default=[])
parser.add_argument('--pin_gpu', type=str_to_bool, default=True)

# forecasting task
parser.add_argument('--seq_len', type=int, default=96, help='input sequence length')
parser.add_argument('--label_len', type=int, default=48, help='start token length')
parser.add_argument('--pred_len', type=int, default=96, help='prediction sequence length')

# DLinear
parser.add_argument('--individual', action='store_true', default=False,
                    help='DLinear: a linear layer for each variate(channel) individually')

# PatchTST
parser.add_argument('--fc_dropout', type=float, default=0.05, help='fully connected dropout')
parser.add_argument('--head_dropout', type=float, default=0.0, help='head dropout')
parser.add_argument('--patch_len', type=int, default=16, help='patch length')
parser.add_argument('--stride', type=int, default=8, help='stride')
parser.add_argument('--padding_patch', default='end', help='None: None; end: padding on the end')
parser.add_argument('--revin', type=int, default=1, help='RevIN; True 1 False 0')
parser.add_argument('--affine', type=int, default=0, help='RevIN-affine; True 1 False 0')
parser.add_argument('--subtract_last', type=int, default=0, help='0: subtract mean; 1: subtract last')
parser.add_argument('--decomposition', type=int, default=0, help='decomposition; True 1 False 0')
parser.add_argument('--kernel_size', type=int, default=25, help='decomposition-kernel')
parser.add_argument('--drop_last', action='store_true', default=False)

# Formers
parser.add_argument('--embed_type', type=int, default=0,
                    help='0: default 1: value embedding + temporal embedding + positional embedding 2: value embedding + temporal embedding 3: value embedding + positional embedding 4: value embedding')
parser.add_argument('--d_model', type=int, default=512, help='dimension of model')
parser.add_argument('--n_heads', type=int, default=8, help='num of heads')
parser.add_argument('--e_layers', type=int, default=2, help='num of encoder layers')
parser.add_argument('--d_layers', type=int, default=1, help='num of decoder layers')
parser.add_argument('--d_ff', type=int, default=2048, help='dimension of fcn')
parser.add_argument('--moving_avg', type=int, default=25, help='window size of moving average')
parser.add_argument('--factor', type=int, default=3, help='attn factor')
parser.add_argument('--distil', action='store_false',
                    help='whether to use distilling in encoder, using this argument means not using distilling',
                    default=True)
parser.add_argument('--dropout', type=float, default=0.05, help='dropout')
parser.add_argument('--embed', type=str, default='timeF',
                    help='time features encoding, options:[timeF, fixed, learned]')
parser.add_argument('--activation', type=str, default='gelu', help='activation')
parser.add_argument('--output_attention', action='store_true', help='whether to output attention in encoder')
parser.add_argument('--output_enc', action='store_true', help='whether to output embedding from encoder')
parser.add_argument('--do_predict', action='store_true', help='whether to predict unseen future data')

# Crossformer
parser.add_argument('--seg_len', type=int, default=24, help='segment length (L_seg)')
parser.add_argument('--win_size', type=int, default=2, help='window size for segment merge')
parser.add_argument('--num_routers', type=int, default=10, help='num of routers in Cross-Dimension Stage of TSA (c)')

# iTransformer
parser.add_argument('--class_strategy', type=str, default='projection', help='projection/average/cls_token')

# MTGNN
parser.add_argument('--subgraph_size', type=int, default=20, help='k')
parser.add_argument('--in_dim', type=int, default=1)

# GPT4TS
parser.add_argument('--gpt_layers', type=int, default=6)
parser.add_argument('--tmax', type=int, default=10)
parser.add_argument('--patch_size', type=int, default=16)

# optimization
parser.add_argument('--num_workers', type=int, default=0, help='data loader num workers')
parser.add_argument('--itr', type=int, default=5, help='experiments times')
parser.add_argument('--train_epochs', type=int, default=100, help='train epochs')
parser.add_argument('--begin_valid_epoch', type=int, default=0)
parser.add_argument('--batch_size', type=int, default=32, help='batch size of train input data')
parser.add_argument('--patience', type=int, default=5, help='early stopping patience')
parser.add_argument('--optim', type=str, default='Adam')
parser.add_argument('--learning_rate', type=float, default=0.0001, help='optimizer learning rate')
parser.add_argument('--des', type=str, default='test', help='exp description')
parser.add_argument('--loss', type=str, default='mse', help='loss function')
parser.add_argument('--lradj', type=str, default='type3', help='adjust learning rate')
parser.add_argument('--use_amp', action='store_true', help='use automatic mixed precision training', default=False)
parser.add_argument('--pct_start', type=float, default=0.3, help='pct_start')
parser.add_argument('--warmup_epochs', type=int, default=5)

# GPU
parser.add_argument('--use_gpu', type=str_to_bool, default=True, help='use gpu')
parser.add_argument('--gpu', type=int, default=0, help='gpu')
parser.add_argument('--use_multi_gpu', action='store_true', help='use multiple gpus', default=False)
parser.add_argument('--devices', type=str, default='0,1,2,3', help='device ids of multile gpus')
parser.add_argument('--test_flop', action='store_true', default=False, help='See utils/tools for usage')
parser.add_argument("--local-rank", default=os.getenv('LOCAL_RANK', -1), type=int)

# SOLID
parser.add_argument('--test_train_num', type=int, default=500)
parser.add_argument('--selected_data_num', type=int, default=5)
parser.add_argument('--lambda_period', type=float, default=0.1)
parser.add_argument('--whole_model', action='store_true')
parser.add_argument('--continual', action='store_true')

args = parser.parse_args()

args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

if args.model.endswith('_Ensemble') and 'TCN' not in args.model and 'FSNet' not in args.model:
    args.model = args.model[:-len('_Ensemble')]
    args.ensemble = True
else:
    args.ensemble = False

import platform

if platform.system() == 'Windows':
    torch.cuda.set_per_process_memory_fraction(48 / 61, 0)

if args.use_gpu and args.use_multi_gpu:
    args.devices = args.devices.replace(' ', '')
    device_ids = args.devices.split(',')
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

args.enc_in, args.c_out = data_settings[args.dataset][args.features]
args.data_path = data_settings[args.dataset]['data']
args.dec_in = args.enc_in
if args.model.endswith('_leak'):
    args.model = args.model[:-len('_leak')]
    args.leakage = True
if args.online_method and args.online_method.endswith('_leak'):
    args.online_method = args.online_method[:-len('_leak')]
    args.leakage = True

if args.tag and args.tag[0] != '_':
    args.tag = '_' + args.tag

args.data = args.data_path[:5] if args.data_path.startswith('ETT') else 'custom'
if args.model.startswith('GPT4TS'):
    if not args.online_method and not args.do_predict:
        args.data += '_CI'
    else:
        if args.dataset == 'ECL':
            args.batch_size = min(args.batch_size, 3)
        elif args.dataset == 'Traffic':
            args.batch_size = 1
if hasattr(args, 'border_type'):
    settings.get_borders(args)

Exp = Exp_Main

args.model_id = f'{args.dataset}_{args.seq_len}_{args.pred_len}_{args.model}'
if args.normalization is not None:
    args.model_id += '_' + args.normalization

if args.border_type == 'online':
    args.patience = min(args.patience, 3)

if args.online_method:
    args.train_epochs = min(args.train_epochs, 25)
    args.save_opt = True
    if 'FSNet' in args.model and args.online_method == 'Online':
        args.online_method = 'FSNet'
    if args.online_method == 'FSNet' and 'TCN' in args.model:
        args.model = args.model.replace('TCN', 'FSNet')

    if args.online_method == 'Online':
        args.pretrain = True
        args.only_test = True

    if 'FSNet' in args.model:
        args.pretrain = False
    elif args.online_method.lower() in settings.peft_methods:
        args.pretrain = True
        args.freeze = True

    Exp = getattr(exps, 'Exp_' + args.online_method)

    if args.online_method == 'SOLID':
        args.pretrain = True
        args.only_test = True
        args.online_method = 'Online'
        if not args.whole_model:
            args.freeze = True

args.timeenc = 2
if args.override_hyper and args.model in settings.hyperparams:
    for k, v in settings.get_hyperparams(args.dataset, args.model, args, args.reduce_bs).items():
        args.__setattr__(k, v)

if args.local_rank != -1:
    torch.cuda.set_device(args.local_rank)
    args.gpu = args.local_rank
    torch.distributed.init_process_group(backend="nccl", init_method='env://')
    args.num_gpus = torch.cuda.device_count()
    args.batch_size = args.batch_size // args.num_gpus

if args.model in ['MTGNN']:
    if 'feat_dim' in data_settings[args.dataset]:
        args.in_dim = data_settings[args.dataset]['feat_dim']
        args.enc_in = int(args.enc_in / args.in_dim)
        if args.features == 'M':
            args.c_out = int(args.c_out / args.in_dim)

if args.model in settings.need_x_mark:
    # args.optim = 'AdamW' if args.optim != 'AdamW' and args.online_method.lower() == 'Concept_Tune' else args.optim
    args.optim = 'AdamW'
    args.patience = 3

args.find_unused_parameters = args.model in ['MTGNN']

data_name = args.data_path.split("/")[-1].split(".")[0]
if platform.system() != 'Windows':
    path = './'
else:
    path = 'D:/data/'
    if args.checkpoints:
        args.checkpoints = 'D:/checkpoints/'

if args.online_method:
    flag = args.online_method.lower()
    if not args.border_type:
        if args.online_method == 'Online':
            flag = args.data
            args.checkpoints = ""
        else:
            flag = args.data + '_' + flag

    if flag == 'fsnet':
        flag = 'online'

    if args.online_method == 'OneNet' and args.pretrain:
        fsnet_name = "FSNet_RevIN"
        args.fsnet_path = f'./checkpoints/{args.dataset}_60_{args.pred_len}_{fsnet_name}_' \
                          f'online_ftM_sl60_ll48_pl{args.pred_len}_lr{settings.pretrain_lr_online_dict[fsnet_name][args.dataset]}' \
                          f'_uniFalse_dm512_nh8_el2_dl1_df2048_fc3_ebtimeF_dtTrue_test_{ii}/checkpoint.pth'

    if 'proceed' in flag:
        if not args.freeze:
            flag += "_fulltune"
        if not args.pretrain:
            flag += "_new"
        flag += f"_{args.lradj}"
        flag += f'_{args.tune_mode}_btl{args.bottleneck_dim}_ema{args.ema}'
        if args.concept_dim:
            flag += f'_mid{args.concept_dim}'
        if not args.individual_generator:
            flag += '_share'
        if args.share_encoder:
            flag += '_share_enc'
        if args.wo_clip:
            flag += '_noclip'
else:
    flag = args.border_type if args.border_type else args.data

print('Args in experiment:')
print(args)

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    # torch.backends.cudnn.benchmark=False
    # torch.backends.cudnn.deterministic = True

if __name__ == '__main__':
    # args = get_args()
    train_data, train_loader, vali_data, vali_loader = None, None, None, None
    test_data, test_loader = None, None

    all_results = {'mse': [], 'mae': []}
    for ii in range(args.itr):
        if ii == 0 and args.skip and os.path.exists(args.skip):
            if args.wo_test:
                continue
            with open(args.skip, 'rt', encoding='utf-8', errors='ignore') as f:
                for line in f.readlines():
                    if line.startswith('mse:'):
                        splits = line.split(',')
                        mse, mae = splits[0].split(':')[1], splits[1].split(':')[1]
                        all_results['mse'].append(float(mse))
                        all_results['mae'].append(float(mae))
                        break
            if len(all_results['mse']) > 0:
                continue
        if args.border_type:
            if args.model in ['PatchTST', 'iTransformer']:
                fix_seed = 2021 + ii
            else:
                fix_seed = 2023 + ii
        else:
            fix_seed = 2023 + ii if args.model == 'iTransformer' else 2021 + ii
        setup_seed(fix_seed)
        print('Seed:', fix_seed)

        setting = '{}_{}_ft{}_sl{}_ll{}_pl{}_lr{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
            args.model_id,
            flag,
            args.features,
            args.seq_len,
            args.label_len,
            args.pred_len,
            args.learning_rate,
            args.d_model,
            args.n_heads,
            args.e_layers,
            args.d_layers,
            args.d_ff,
            args.factor,
            args.embed,
            args.distil,
            args.des, ii)

        if args.pretrain:
            pretrain_lr = settings.pretrain_lr_online_dict[args.model + ("_RevIN" if args.normalization else "")][args.dataset] \
                if args.online_method else settings.pretrain_lr_dict[args.model][args.dataset]
            if not args.border_type and args.model == 'iTransformer' and args.dataset == 'Weather':
                pretrain_lr = 0.0001
            pretrain_setting = '{}_{}_ft{}_sl{}_ll{}_pl{}_lr{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
                args.model_id,
                args.border_type if args.border_type else args.data,
                args.features,
                args.seq_len,
                args.label_len,
                args.pred_len,
                pretrain_lr,
                args.d_model,
                args.n_heads,
                args.e_layers,
                args.d_layers,
                args.d_ff,
                args.factor,
                args.embed,
                args.distil,
                args.des, ii)
            args.pred_path = os.path.join('./results/', pretrain_setting, 'real_prediction.npy')
            if platform.system() == 'Windows':
                args.load_path = os.path.join('D://checkpoints/', pretrain_setting, 'checkpoint.pth')
            else:
                args.load_path = os.path.join('./checkpoints/', pretrain_setting, 'checkpoint.pth')
        exp = Exp(args)  # set experiments

        if train_data is None:
            train_data, train_loader = exp._get_data('train')
        if not hasattr(args, 'borders'):
            args.borders = train_data.borders
            if args.border_type != 'online' and args.model == 'PatchTST':
                settings.drop_last_PatchTST(args) # SOLID dropout the last when data split = 7:2:1
        exp.wrap_data_kwargs['borders'] = args.borders

        path = os.path.join(args.checkpoints, setting, 'checkpoint.pth')
        if args.online_method not in ['Online', 'SOLID', 'ER', 'DERpp']:
            print('Checkpoints in', path)
            if (args.only_test or args.do_valid) and os.path.exists(path):
                print('Loading', path)
                exp.load_checkpoint(path)
                print('Learning rate of model_optim is', exp.model_optim.param_groups[0]['lr'])
            else:
                print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
                _, train_data, train_loader, vali_data, vali_loader = exp.train(setting, train_data, train_loader,
                                                                                vali_data, vali_loader)
                torch.cuda.empty_cache()

        if args.online_learning_rate is not None and not isinstance(exp, Exp_SOLID):
            for j in range(len(exp.model_optim.param_groups)):
                exp.model_optim.param_groups[j]['lr'] = args.online_learning_rate
            print('Adjust learning rate of model_optim to', exp.model_optim.param_groups[0]['lr'])

        if args.do_valid and args.online_method and args.local_rank <= 0:
            assert isinstance(exp, Exp_Online)
            mse, mae = exp.online(online_data=vali_data if isinstance(vali_data, Dataset_Recent) else None,
                                  phase='val', show_progress=True)[:2]
            print('Best Valid MSE:', mse)
            all_results['mse'].append(mse)
            all_results['mae'].append(mae)
            continue

        if args.do_predict:
            print('>>>>>>>predicting : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
            setup_seed(fix_seed)
            mse, mae = exp.predict(path, setting, True)[:2]
            all_results['mse'].append(mse)
            all_results['mae'].append(mae)
        elif not args.wo_test and not args.train_only and args.local_rank <= 0:
            print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
            if isinstance(exp, Exp_Online):
                setup_seed(fix_seed)
                if not isinstance(exp, Exp_SOLID) and not args.wo_valid:
                    vali_data = None
                    torch.cuda.empty_cache()
                    gc.collect()
                    exp.update_valid()
                mse, mae, test_data = exp.online(test_data)
            else:
                mse, mae, test_data, test_loader = exp.test(setting, test_data, test_loader)
            all_results['mse'].append(mse)
            all_results['mae'].append(mae)
        torch.cuda.empty_cache()

    for k in all_results.keys():
        all_results[k] = np.array(all_results[k])
        all_results[k] = [all_results[k].mean(), all_results[k].std()]
    pprint(all_results)

Overwriting run.py
